[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C02/E02-3_Ressuage.ipynb)

# Défaut rare au contrôle par ressuage : rappel, précision et F-score

## Le contexte

Des pièces forgées, soudées ou usinées, après un certain nombre de cycles de service, passent au contrôle par ressuage : un liquide pénètre dans les fissures débouchantes et un révélateur les fait apparaître. Sur les 600 pièces de ce relevé, seules 53 montrent une indication, soit environ une sur onze.

Le contrôle immobilise la pièce et coûte du temps. On voudrait cibler les pièces à contrôler en priorité, d'après leur procédé d'élaboration, leur contrainte de service, leur nombre de cycles et leur état de surface. Laisser passer une pièce fissurée coûte bien plus cher que contrôler une pièce saine : le modèle doit donc retrouver les défauts, quitte à lever quelques fausses alertes.

C'est un problème de classification binaire sur une classe rare : la grandeur à prédire est une catégorie, `1` pour une indication, `0` sinon.

Défi : prédire `indication` et obtenir un F-score macro supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est une régression logistique sur deux grandeurs seulement, la contrainte de service et le nombre de cycles, au seuil habituel de 0,5 ; il atteint un F-score macro de 0,691 : à vous d'ajouter les autres colonnes et de faire mieux, en surveillant le rappel. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Ressuage.xlsx`, dans le dossier `donnees` du dépôt : 600 pièces, 6 colonnes. Une ligne est une pièce.

| Colonne | Signification | Unité |
|---|---|---|
| `procede` | procédé d'élaboration : `forge`, `soude` ou `usine` | catégorie |
| `contrainte_MPa` | contrainte de service | MPa |
| `cycles_milliers` | nombre de cycles subis | milliers |
| `rugosite_Ra_um` | rugosité de surface | µm |
| `temps_penetration_min` | temps de pénétration du produit | min |
| `indication` | cible : `1` si le ressuage révèle une indication, `0` sinon | 0 ou 1 |

Ce jeu est simulé : une fissure apparaît quand le nombre de cycles dépasse la tenue en fatigue de la pièce, qui suit une loi en puissance de la contrainte avec une dispersion importante, et le ressuage la détecte presque toujours mais peut aussi signaler à tort une surface rugueuse. Quelques cases sont vides, comme sur un vrai relevé. Il enseigne une méthode et ne remplace pas une étude de fiabilité.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la rareté de la classe à prédire avant toute modélisation. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Ressuage.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Ressuage.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "indication"
print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides par colonne :")
print(vides[vides > 0].to_string())
print(df[cible].value_counts().to_string())
print(f"part des pièces avec indication : {df[cible].mean():.1%}")
df.head(3)

Lecture du résultat : 600 pièces, 6 colonnes. Deux colonnes ont des cases vides : `cycles_milliers` (32) et `rugosite_Ra_um` (17). Seules 53 pièces ont une indication, soit 8,8 % : la classe à prédire est rare.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Ressuage.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : séparer l'apprentissage et le test

Un quart des pièces est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la grandeur à prédire. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Le tirage est stratifié, ce qui est indispensable ici : sans cela, le test pourrait ne contenir que très peu de pièces avec indication, voire aucune.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=GRAINE, stratify=y)
print("apprentissage :", len(X_train), "pièces, dont", int(y_train.sum()), "avec indication")
print("test :", len(X_test), "pièces, dont", int(y_test.sum()), "avec indication")

### Étape 3 : regarder où se trouvent les indications

On calcule la part de pièces avec indication par procédé, puis par tranche de contrainte de service. Ces parts croisent la grandeur à prédire : elles se lisent sur l'apprentissage seul, sans la base de test.

In [ ]:
appr = df.loc[X_train.index]
par_procede = appr.groupby("procede")[cible].mean()
print((100 * par_procede).round(1).to_string())

tranches = pd.cut(appr["contrainte_MPa"], bins=[60, 90, 120, 150, 180], include_lowest=True)
par_tranche = appr.groupby(tranches, observed=True)[cible].mean()

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.bar(range(len(par_tranche)), 100 * par_tranche.values, color="0.55", edgecolor="black", hatch="//")
ax.set_xticks(range(len(par_tranche)))
ax.set_xticklabels([f"{round(t.left)} à {round(t.right)}" for t in par_tranche.index])
ax.set_xlabel("contrainte de service (MPa)")
ax.set_ylabel("pièces avec indication (%)")
ax.grid(alpha=0.3, axis="y")
plt.show()

Lecture du résultat : sur les 450 pièces d'apprentissage, la part d'indications dépend du procédé : 22,3 % pour les pièces soudées, 6,7 % pour les forgées et 0,5 % pour les usinées. Elle monte aussi avec la contrainte : 0,8 % sous 90 MPa, 24 % dans la dernière tranche. Les indications se concentrent donc sur quelques profils de pièces, ce qui rend le tri possible.

### Étape 4 : le piège de l'exactitude

Un témoin qui répond toujours « pas d'indication » ne détecte jamais rien. On compare son exactitude (part de réponses justes), son rappel (part des pièces avec indication qu'il retrouve) et son F-score macro.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score

y_temoin = np.zeros(len(y_test), dtype=int)
exactitude_temoin = accuracy_score(y_test, y_temoin)
rappel_temoin = recall_score(y_test, y_temoin, zero_division=0)
f1_temoin = f1_score(y_test, y_temoin, average="macro", zero_division=0)
print(f"exactitude du témoin : {exactitude_temoin:.3f}")
print(f"rappel du témoin : {rappel_temoin:.3f}")
print(f"F-score macro du témoin : {f1_temoin:.3f}")

Lecture du résultat : le témoin a une exactitude de 0,913, qui semble excellente, et un rappel nul : il ne retrouve aucune pièce fissurée. Sur une classe rare, l'exactitude récompense l'inaction. Le F-score macro tombe à 0,477 parce qu'il compte la classe rare pour moitié.

### Étape 5 : construire le modèle de départ

Le modèle de départ est volontairement simple : une régression logistique, qui estime la probabilité d'une indication, puis déclare « indication » toute pièce dont la probabilité dépasse 0,5. Il n'utilise que les deux grandeurs de la fatigue, la contrainte de service et le nombre de cycles. Un pipeline remplit d'abord les cases vides par la médiane de l'apprentissage, puis met les deux mesures à l'échelle. Il apprend sur `X_train` seul. Les autres colonnes, dont le procédé, sont laissées de côté : elles sont votre marge de progression.

In [ ]:
variables = ["contrainte_MPa", "cycles_milliers"]

# Votre code :

### Étape 6 : mesurer, puis lire le rappel et la précision

Sur la classe rare, deux questions se posent. Parmi les pièces déclarées avec indication, quelle part l'est vraiment ? C'est la précision. Parmi les pièces qui ont vraiment une indication, quelle part est retrouvée ? C'est le rappel. Le F-score d'une classe est la moyenne harmonique des deux, et le F-score macro est la moyenne des F-scores des deux classes.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_score

y_pred = modele.predict(X_test[variables])
score_depart = f1_score(y_test, y_pred, average="macro")
precision_depart = precision_score(y_test, y_pred, zero_division=0)
rappel_depart = recall_score(y_test, y_pred)
print(f"F-score macro : {score_depart:.3f}")
print(f"précision de la classe indication : {precision_depart:.3f}")
print(f"rappel de la classe indication : {rappel_depart:.3f}")
print(classification_report(y_test, y_pred, digits=3))
matrice = confusion_matrix(y_test, y_pred, labels=[1, 0])
print("matrice de confusion (lignes : réel, colonnes : prédit ; indication, pas d'indication)")
print(matrice)

Lecture du résultat : le F-score macro est de 0,691, au-dessus du témoin (0,477). Le détail est plus parlant. La précision est de 0,667 : 6 alertes, dont 2 fausses. Le rappel est de 0,308 : 4 pièces fissurées retrouvées sur 13, et 9 laissées passer. Le modèle donne le bénéfice du doute à la pièce : dans l'incertitude, il la déclare saine. Pour un contrôle de sécurité, ce n'est pas le bon côté du doute. C'est ce chiffre, mesuré sur des pièces que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : toutes les colonnes, puis déplacer le doute vers la sécurité

Le modèle de départ de l'étape 5 reste le score de référence du défi. Les deux cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première compare par validation croisée le modèle de départ, le même type de modèle sur toutes les colonnes, puis avec `class_weight="balanced"`, qui pèse davantage les erreurs sur la classe rare. La seconde compare des candidats, retient le meilleur et fixe le seuil de décision sur des probabilités obtenues par validation croisée, avec une règle métier : retrouver au moins 80 % des défauts, avec la meilleure précision possible. Le test ne sert qu'une fois, à la fin. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le F-score macro du modèle de départ de l'étape 5 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = f1_score(y_test, modele.predict(X_test[variables]), average="macro")
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE f1_macro = %.3f" % score)

## Synthèse

1. Sur une classe rare, l'exactitude récompense l'inaction : un témoin qui ne détecte rien paraît excellent.
2. La précision répond à « parmi mes alertes, combien sont justes ? », le rappel à « parmi les défauts, combien ai-je retrouvés ? ». Le F-score les réunit en une moyenne harmonique, le F-score macro moyenne les deux classes.
3. Le seuil de 0,5 donne le bénéfice du doute à la classe majoritaire. Quand manquer un défaut coûte plus cher qu'une fausse alerte, on le déplace, et la règle métier se traduit en un rappel minimal.
4. Le seuil, comme le modèle, se choisit par validation croisée sur l'apprentissage, jamais sur le test.
5. Un tirage stratifié garde la classe rare dans les deux bases ; le score d'une base de test qui ne compte qu'une dizaine de défauts bouge d'un tirage à l'autre.